# 🤗 x 🦾: Training ACT with LeRobot Notebook

Welcome to the **LeRobot ACT training notebook**! This notebook provides a ready-to-run setup for training imitation learning policies using the [🤗 LeRobot](https://github.com/huggingface/lerobot) library.

In this example, we train an `ACT` policy using a dataset hosted on the [Hugging Face Hub](https://huggingface.co/), and optionally track training metrics with [Weights & Biases (wandb)](https://wandb.ai/).

## ⚙️ Requirements
- A Hugging Face dataset repo ID containing your training data (`--dataset.repo_id=YOUR_USERNAME/YOUR_DATASET`)
- Optional: A [wandb](https://wandb.ai/) account if you want to enable training visualization
- Recommended: GPU runtime (e.g., NVIDIA A100) for faster training

## ⏱️ Expected Training Time
Training with the `ACT` policy for 100,000 steps typically takes **about 1.5 hours on an NVIDIA A100** GPU. On less powerful GPUs or CPUs, training may take significantly longer.

## Example Output
Model checkpoints, logs, and training plots will be saved to the specified `--output_dir`. If `wandb` is enabled, progress will also be visualized in your wandb project dashboard.


# Setup
- define system variables
- Update python to 3.12
- Install dependencies

## Install conda
This cell uses `condacolab` to bootstrap a full Conda environment inside Google Colab.


In [ ]:
# Cell 1 — Fresh start, run alone, do NOT run anything else before this
# Install condacolab to manage Conda environments in Google Colab.
# This command installs the `condacolab` library, which allows Conda environments to be managed within Google Colab.
!pip install -q condacolab
import condacolab
# Install Miniconda with Python 3.12, which automatically restarts the kernel.
# This command installs Miniconda with Python 3.12 and automatically restarts the kernel, setting up the Conda environment.
condacolab.install_from_url(
    "https://repo.anaconda.com/miniconda/Miniconda3-py312_24.5.0-0-Linux-x86_64.sh"
)
# Kernel will restart automatically

# Add dist-packages to Python's path permanently via a .pth file.
# This command ensures that Python packages installed via `pip` are discoverable by the new Conda-managed Python environment.
!echo "/usr/local/lib/python3.12/dist-packages" > /usr/local/lib/python3.12/site-packages/distpackages.pth

# Verify torch is now found after the environment setup.
# This command verifies the PyTorch installation and its version after the environment setup.
!python -c "import torch; print(torch.__version__)"

# Upgrade pip to the latest version to avoid potential dependency conflicts and ensure access to new features.
# This command upgrades pip to its latest version to prevent potential dependency conflicts and access new features.
!pip install --upgrade pip

# Install a specific version of pyarrow to satisfy potential dependency requirements of other libraries (e.g., lerobot).
# This command installs a specific version of `pyarrow` to satisfy potential dependency requirements of other libraries like `lerobot`.
!pip install "pyarrow==17.0.0"

# Re-add dist-packages to Python's path permanently via a .pth file, ensuring consistency after pip installations.
# This command re-adds `dist-packages` to Python's path, ensuring consistency after pip installations.
!echo "/usr/local/lib/python3.12/dist-packages" > /usr/local/lib/python3.12/site-packages/distpackages.pth

In [ ]:
 # After `condacolab.install_from_url` restarts the kernel, the Conda environment is active.
 # The `condacolab` library itself is generally not needed or accessible in the new Conda environment.
 # The following commands verify the new Conda environment's setup directly.

 # Confirm the active Python version. If `condacolab` successfully switched the environment,
 # this should report Python 3.12, indicating the Conda-managed Python is active.
 !python --version   # Must say 3.12

 # Show the full path to the currently active Python executable.
 # This helps confirm that the system is using the Python installed by Conda (usually in /usr/local/bin).
 !which python

 # Show the full path to the currently active pip executable.
 # This verifies that pip is associated with the Conda-managed Python environment.
 !which pip

 # Verify the PyTorch installation and its version after the environment setup.
 # This command imports torch and prints its version, confirming that PyTorch is installed and accessible in the current Python environment.
 !python -c "import torch; print(torch.__version__)"

In [ ]:
# Clone the LeRobot repository from GitHub.
!git clone https://github.com/huggingface/lerobot.git
# Install a specific version of FFmpeg, which is likely required for video processing within LeRobot, specifying the conda-forge channel.
!conda install ffmpeg=7.1.1 -c conda-forge
# Change the directory to 'lerobot' and install the library in editable mode, allowing local changes to be immediately reflected.
!cd lerobot && pip install -e .
# Install additional Python packages with specific version constraints required by LeRobot.
!pip install "tqdm>=4.67" "pluggy>=1.5" "requests>=2.32.4"

In [ ]:
HF_USER = "XXXXX"  # Defines your Hugging Face username, used for identifying your account.
HF_TOKEN = "hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX" # Stores your Hugging Face access token for authentication, allowing programmatic access to the Hub.
HF_REPO_NAME = "EE5108_batch_sim" # Defines the name of the Hugging Face dataset repository, used to identify the specific dataset.
# WANDB_PROJECT = "EE5108_project" # Optional: Defines the Weights & Biases project name if wandb is enabled for experiment tracking.

In [2]:
from huggingface_hub import login # Import the 'login' utility from the Hugging Face Hub library.

# Perform a non-interactive login to the Hugging Face Hub.
# This uses the 'HF_TOKEN' variable, which was defined earlier, for authentication.
# This step is crucial for programmatic access to Hugging Face Hub features,
# such as downloading datasets or uploading models.
login(token=HF_TOKEN)

## Install LeRobot
This cell clones the `lerobot` repository from Hugging Face, installs FFmpeg (version 7.1.1), and installs the package in editable mode.


## Weights & Biases login
This cell logs you into Weights & Biases (wandb) to enable experiment tracking and logging.

In [3]:
import os

# This cell handles Weights & Biases (wandb) login for experiment tracking.
# The `wandb login` command is currently commented out because wandb logging
# is explicitly disabled in the training command (`--wandb.enable=False`).
#
# If you wish to enable Weights & Biases logging for visualizing training progress,
# you would need to:
# 1. Uncomment the `!wandb login` line below.
# 2. Set `--wandb.enable=True` in the training command (Cell `Ufss6US6xbpi`).
# 3. Ensure you have a Weights & Biases account and are authenticated.
#
# To log in to Weights & Biases interactively (uncomment and run):
# !wandb login

## Start training ACT with LeRobot

This cell runs the `train.py` script from the `lerobot` library to train a robot control policy.  

Make sure to adjust the following arguments to your setup:

1. `--dataset.repo_id=YOUR_HF_USERNAME/YOUR_DATASET`:  
   Replace this with the Hugging Face Hub repo ID where your dataset is stored, e.g., `pepijn223/il_gym0`.

2. `--policy.type=act`:  
   Specifies the policy configuration to use. `act` refers to [configuration_act.py](../lerobot/common/policies/act/configuration_act.py), which will automatically adapt to your dataset’s setup (e.g., number of motors and cameras).

3. `--output_dir=outputs/train/...`:  
   Directory where training logs and model checkpoints will be saved.

4. `--job_name=...`:  
   A name for this training job, used for logging and Weights & Biases.

5. `--policy.device=cuda`:  
   Use `cuda` if training on an NVIDIA GPU. Use `mps` for Apple Silicon, or `cpu` if no GPU is available.

6. `--wandb.enable=true`:  
   Enables Weights & Biases for visualizing training progress. You must be logged in via `wandb login` before running this. Set to `False` if you do not plan on using Weights & Biases.

In [ ]:
import huggingface_hub # Import the Hugging Face Hub library, which provides tools to interact with the Hugging Face Hub.
print(huggingface_hub.__version__) # Print the installed version of the huggingface_hub library to verify that the correct version is installed and accessible.

In [ ]:
from huggingface_hub import snapshot_download # Import the function to download a snapshot of a repository.

# This cell downloads a snapshot of the specified dataset from the Hugging Face Hub
# into the local Colab environment. This is crucial for accessing the training data.
snapshot_download(
    repo_id=f"{HF_USER}/{HF_REPO_NAME}", # `repo_id` specifies the dataset to download. It's constructed
                                        # using the previously defined `HF_USER` (Hugging Face username)
                                        # and `HF_REPO_NAME` (the name of the dataset repository).
    repo_type="dataset", # `repo_type` is set to 'dataset' to explicitly indicate that the
                         # repository being downloaded contains a dataset.
    local_dir=f"/root/.cache/huggingface/lerobot/{HF_USER}/{HF_REPO_NAME}" # `local_dir` defines the target
                                                                           # path within the Colab environment where
                                                                           # the dataset snapshot will be stored.
                                                                           # This path is configured to use the
                                                                           # Hugging Face cache directory structure.
)

In [ ]:
!cd lerobot && python src/lerobot/scripts/lerobot_train.py \
  --dataset.repo_id="{HF_USER}/{HF_REPO_NAME}" \
  --policy.type=act \
  --output_dir=outputs/train/hf_act_record0 \
  --job_name=hf_act_training_job \
  --policy.device=cuda \
  --wandb.enable=False \
  --policy.repo_id=BrianMIR1/hf_act_recordpolicy0

## Post-Training Hugging Face Hub Operations

After training is complete, this section guides you through uploading the generated model checkpoint to the Hugging Face Hub. Ensure you have already authenticated with the Hugging Face Hub in the previous setup steps.

In [ ]:
from huggingface_hub import HfApi

HF_USERNAME = HF_USER
HF_REPO_NAME = "act-configs"

api = HfApi()
repo_id = f"{HF_USERNAME}/{HF_REPO_NAME}"
files_in_repo = api.list_repo_files(repo_id=repo_id)

print(f"Files in {repo_id}:")
for file in files_in_repo:
    print(f"- {file}")

### Configure Hugging Face Token

Add your HF_TOKEN (AKA Secret) to Google Colab to enable Colab to access your HF repositories. This is optional and might need modification if you are using another cloud provider.

In [ ]:
from google.colab import userdata
import os

# Use the HF_TOKEN variable defined earlier in the notebook
os.environ["HF_TOKEN"] = HF_TOKEN

# Verify token is loaded (optional)
if os.getenv("HF_TOKEN"):
    print("Hugging Face token loaded successfully.")
else:
    print("Error: Hugging Face token not found. Please ensure HF_TOKEN variable is set.")

In [ ]:
from google.colab import userdata
userdata.get('HF_TOKEN')

### Download files from Hugging Face Hub into Colab

Now, you can use `hf_hub_download` to pull the files directly to your Colab environment. You can then download them from Colab to your local machine. This is optional.

In [ ]:
from huggingface_hub import hf_hub_download

# Your Hugging Face repository details
HF_CONFIG_REPO_ID = f"{HF_USER}/act-configs" # Where train_config.json should be
HF_POLICY_REPO_ID = f"{HF_USER}/hf_act_recordpolicy0" # Where the trained model is

# Define the files to download
config_file_name = "train_config.json"
model_file_name = "model.safetensors"
tokenizer_processor_file_name = "policy_preprocessor_step_3_normalizer_processor.safetensors" # Corrected filename

# Download train_config.json
train_config_path = hf_hub_download(repo_id=HF_CONFIG_REPO_ID, filename=config_file_name)
print(f"Downloaded {config_file_name} to: {train_config_path}")

# Download the trained model files
model_path = hf_hub_download(repo_id=HF_POLICY_REPO_ID, filename=model_file_name)
print(f"Downloaded {model_file_name} to: {model_path}")

tokenizer_processor_path = hf_hub_download(repo_id=HF_POLICY_REPO_ID, filename=tokenizer_processor_file_name)
print(f"Downloaded {tokenizer_processor_file_name} to: {tokenizer_processor_path}")

print("\nAll specified files have been downloaded to your Colab environment.")
print("You can find them in the paths printed above. To download them to your local machine, right-click on the files in the Colab file browser (left sidebar) and select 'Download'.")

### Verify `train_config.json` existence locally. This is needed for restarting training and testing of the policy.

In [ ]:
!ls -l /content/lerobot/outputs/train/hf_act_record0/

In [ ]:
HF_USERNAME = HF_USER
HF_REPO_NAME = "act-configs"

!hf repo-files $HF_USERNAME/$HF_REPO_NAME

In [ ]:
!hf auth login

In [ ]:
!hf upload {HF_USER}/hf_act_record0 \
  /content/lerobot/outputs/train/hf_act_record0/checkpoints/last/pretrained_model

In [ ]:
!hf auth login

### Create a new repository on Hugging Face Hub

This command will create a new repository under your Hugging Face account.

In [ ]:
HF_USERNAME = HF_USER
HF_REPO_NAME = "act-configs"

!hf repo create $HF_REPO_NAME --type model --private

In [ ]:
HF_USERNAME = HF_USER
HF_REPO_NAME = "act-configs"
LOCAL_CONFIG_PATH = "/content/lerobot/outputs/train/hf_act_record0/checkpoints/last/pretrained_model/train_config.json"

!hf upload $HF_USERNAME/$HF_REPO_NAME "$LOCAL_CONFIG_PATH" train_config.json

In [ ]:
from google.colab import userdata
import os

# Use the HF_TOKEN variable defined earlier in the notebook
os.environ["HF_TOKEN"] = HF_TOKEN

# Verify token is loaded (optional)
if os.getenv("HF_TOKEN"):
    print("Hugging Face token loaded successfully.")
else:
    print("Error: Hugging Face token not found. Please ensure HF_TOKEN variable is set.")

In [ ]:
# Verify train_config.json existence locally. This is needed for restarting training and testing of the policy.
!ls -lR /content/lerobot/outputs/train/hf_act_record0/

In [ ]:
from huggingface_hub import HfApi

HF_USERNAME = HF_USER
HF_REPO_NAME = "act-configs"

api = HfApi()
repo_id = f"{HF_USERNAME}/{HF_REPO_NAME}"
files_in_repo = api.list_repo_files(repo_id=repo_id)

print(f"Files in {repo_id}:")
for file in files_in_repo:
    print(f"- {file}")

In [ ]:
from huggingface_hub import hf_hub_download

# Your Hugging Face repository details
HF_CONFIG_REPO_ID = f"{HF_USER}/act-configs" # Where train_config.json should be
HF_POLICY_REPO_ID = f"{HF_USER}/hf_act_recordpolicy0" # Where the trained model is

# Define the files to download
config_file_name = "train_config.json"
model_file_name = "model.safetensors"
tokenizer_processor_file_name = "policy_preprocessor_step_3_normalizer_processor.safetensors" # Corrected filename

# Download train_config.json
train_config_path = hf_hub_download(repo_id=HF_CONFIG_REPO_ID, filename=config_file_name)
print(f"Downloaded {config_file_name} to: {train_config_path}")

# Download the trained model files
model_path = hf_hub_download(repo_id=HF_POLICY_REPO_ID, filename=model_file_name)
print(f"Downloaded {model_file_name} to: {model_path}")

tokenizer_processor_path = hf_hub_download(repo_id=HF_POLICY_REPO_ID, filename=tokenizer_processor_file_name)
print(f"Downloaded {tokenizer_processor_file_name} to: {tokenizer_processor_path}")

print("\nAll specified files have been downloaded to your Colab environment.")
print("You can find them in the paths printed above. To download them to your local machine, right-click on the files in the Colab file browser (left sidebar) and select 'Download'.")

In [ ]:
from huggingface_hub import hf_hub_download

# Your Hugging Face repository details
HF_CONFIG_REPO_ID = f"{HF_USER}/act-configs" # Where train_config.json should be
HF_POLICY_REPO_ID = f"{HF_USER}/hf_act_recordpolicy0" # Where the trained model is

# Define the files to download
config_file_name = "train_config.json"
model_file_name = "model.safetensors"
tokenizer_processor_file_name = "policy_preprocessor_step_3_normalizer_processor.safetensors" # Corrected filename

# Download train_config.json
train_config_path = hf_hub_download(repo_id=HF_CONFIG_REPO_ID, filename=config_file_name)
print(f"Downloaded {config_file_name} to: {train_config_path}")

# Download the trained model files
model_path = hf_hub_download(repo_id=HF_POLICY_REPO_ID, filename=model_file_name)
print(f"Downloaded {model_file_name} to: {model_path}")

tokenizer_processor_path = hf_hub_download(repo_id=HF_POLICY_REPO_ID, filename=tokenizer_processor_file_name)
print(f"Downloaded {tokenizer_processor_file_name} to: {tokenizer_processor_path}")

print("\nAll specified files have been downloaded to your Colab environment.")
print("You can find them in the paths printed above. To download them to your local machine, right-click on the files in the Colab file browser (left sidebar) and select 'Download'.")

In [ ]:
from huggingface_hub import HfApi

HF_USERNAME = HF_USER
HF_REPO_NAME = "act-configs"

api = HfApi()
repo_id = f"{HF_USERNAME}/{HF_REPO_NAME}"
files_in_repo = api.list_repo_files(repo_id=repo_id)

print(f"Files in {repo_id}:")
for file in files_in_repo:
    print(f"- {file}")

In [ ]:
from huggingface_hub import hf_hub_download

# Your Hugging Face repository details
HF_CONFIG_REPO_ID = f"{HF_USER}/act-configs" # Where train_config.json should be
HF_POLICY_REPO_ID = f"{HF_USER}/hf_act_recordpolicy0" # Where the trained model is

# Define the files to download
config_file_name = "train_config.json"
model_file_name = "model.safetensors"
tokenizer_processor_file_name = "policy_preprocessor_step_3_normalizer_processor.safetensors" # Corrected filename

# Download train_config.json
train_config_path = hf_hub_download(repo_id=HF_CONFIG_REPO_ID, filename=config_file_name)
print(f"Downloaded {config_file_name} to: {train_config_path}")

# Download the trained model files
model_path = hf_hub_download(repo_id=HF_POLICY_REPO_ID, filename=model_file_name)
print(f"Downloaded {model_file_name} to: {model_path}")

tokenizer_processor_path = hf_hub_download(repo_id=HF_POLICY_REPO_ID, filename=tokenizer_processor_file_name)
print(f"Downloaded {tokenizer_processor_file_name} to: {tokenizer_processor_path}")

print("\nAll specified files have been downloaded to your Colab environment.")
print("You can find them in the paths printed above. To download them to your local machine, right-click on the files in the Colab file browser (left sidebar) and select 'Download'.")

In [ ]:
from huggingface_hub import HfApi

HF_USERNAME = HF_USER
HF_REPO_NAME = "act-configs"

api = HfApi()
repo_id = f"{HF_USERNAME}/{HF_REPO_NAME}"
files_in_repo = api.list_repo_files(repo_id=repo_id)

print(f"Files in {repo_id}:")
for file in files_in_repo:
    print(f"- {file}")

In [ ]:
from huggingface_hub import hf_hub_download

# Your Hugging Face repository details
HF_CONFIG_REPO_ID = f"{HF_USER}/act-configs" # Where train_config.json should be
HF_POLICY_REPO_ID = f"{HF_USER}/hf_act_recordpolicy0" # Where the trained model is

# Define the files to download
config_file_name = "train_config.json"
model_file_name = "model.safetensors"
tokenizer_processor_file_name = "policy_preprocessor_step_3_normalizer_processor.safetensors" # Corrected filename

# Download train_config.json
train_config_path = hf_hub_download(repo_id=HF_CONFIG_REPO_ID, filename=config_file_name)
print(f"Downloaded {config_file_name} to: {train_config_path}")

# Download the trained model files
model_path = hf_hub_download(repo_id=HF_POLICY_REPO_ID, filename=model_file_name)
print(f"Downloaded {model_file_name} to: {model_path}")

tokenizer_processor_path = hf_hub_download(repo_id=HF_POLICY_REPO_ID, filename=tokenizer_processor_file_name)
print(f"Downloaded {tokenizer_processor_file_name} to: {tokenizer_processor_path}")

print("\nAll specified files have been downloaded to your Colab environment.")
print("You can find them in the paths printed above. To download them to your local machine, right-click on the files in the Colab file browser (left sidebar) and select 'Download'.")